# LangGraph 도구(Tool) 활용: ToolNode 와 에이전트

## 이 노트북의 역할
LLM 이 **도구(함수)를 스스로 골라 호출**하는 에이전트를 만드는 방법을 익힙니다. 먼저 `ToolNode` 로 직접 그래프를 구성해 원리를 이해하고, 그 다음 `create_agent` 한 줄로 같은 일을 하는 방법을 봅니다.

| 단계 | 내용 | 배우는 것 |
|---|---|---|
| 1. 도구 정의 | `@tool` 함수, `bind_tools` | LLM 이 `tool_calls` 를 만드는 방식 |
| 2. ToolNode | `AIMessage.tool_calls` 를 실행해 `ToolMessage` 반환 | LLM 없이도 ToolNode 동작 확인 |
| 3. 직접 구성한 에이전트 | `agent` ⇄ `tools` 루프 + `add_conditional_edges` | ReAct 루프의 구조 |
| 4. `create_agent` | 위 구조를 한 줄로 생성 | 실무에서 쓰는 방법 |
| 5. 복합 도구 실습 | 복리·세금·요일 도구 | 도구 선택·연쇄 호출 |

## 에이전트 루프 (3단계 그래프)
```text
START → agent ──(tool_calls 있음)──→ tools ──┐
          ↑                                   │
          └───────────────────────────────────┘
          └──(tool_calls 없음)──→ END
```
* `agent` 노드: LLM 이 답하거나 도구 호출(`tool_calls`)을 결정합니다.
* `tools` 노드(`ToolNode`): 요청된 도구를 실행하고 결과를 `ToolMessage` 로 상태에 추가합니다.
* 분기는 노드가 아니라 **엣지의 조건**(`should_continue`)입니다.

> 이전 노트북(`01_StateGraph`)은 개발자가 경로를 **고정**한 워크플로우, 이 노트북은 LLM 이 경로를 **스스로 결정**하는 에이전트입니다. 메시지 상태(`MessagesState`)는 `02_MessageState_Basic` 에서 다룹니다.

## 사용 모델 (`get_llm()`)
- `PROVIDER` 값(`"groq"` / `"openai"` / `"upstage"`, 기본값 `"upstage"`)만 바꾸면 이후 모든 셀의 LLM 이 바뀝니다. 바꾼 뒤에는 LLM 설정 셀부터 다시 실행하세요.
- 도구 호출은 **모델에 따라 품질이 다릅니다.** 연쇄 호출(`3+5` 후 `×8`)이 어색하게 나오면 다른 `PROVIDER` 로 바꿔 비교해 보세요.
- LLM 호출은 사용 요금이 발생할 수 있습니다.

`(1) API 키와 LLM 준비`

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 제공자별 API 키 (키 값은 출력하지 않음) — 사용할 제공자의 키만 있으면 됨
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

`(3) LLM 설정 (Groq / OpenAI / Upstage 선택)`
- `PROVIDER` 값만 바꾸면 이후 모든 셀의 모델이 바뀝니다. (Groq 는 OpenAI 호환 API 라서 `ChatOpenAI` 에 `base_url` 만 다르게 연결)
- 바꾼 뒤에는 이 셀부터 다시 실행하세요. OpenAI 선택 시 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "upstage"

# 제공자별 설정 (cls: 사용할 LangChain 채팅 모델 클래스)
LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o-mini",  # 테스트에는 작은 모델 사용 (대안: "gpt-4o")
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: 모델 클래스에 그대로 전달할 추가 옵션 (예: seed=42, timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    model_cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = model_cls(**config, temperature=temperature, max_retries=MAX_RETRIES, **kwargs)
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 도구 호출은 결과가 일정해야 하므로 temperature=0 사용
llm = get_llm(temperature=0)

`(2) 도구 정의`
* `@tool` 데코레이터를 붙인 함수가 도구가 됩니다. **함수 이름·인자 타입·docstring** 이 LLM 에게 전달되는 도구 설명이므로 docstring 을 정확히 씁니다.

In [ ]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """숫자 a와 b를 더합니다."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱합니다."""
    return a * b

In [ ]:
# 도구는 일반 함수처럼 단독 호출할 수 있음: 인자는 딕셔너리로 전달
result = multiply.invoke({"a": 3, "b": 5})

print(type(result))
result

`(3) bind_tools 와 ToolNode`
* `llm.bind_tools(도구 목록)`: LLM 이 도구를 호출할 수 있도록 도구 설명을 함께 전달하는 모델을 만듭니다.
* `ToolNode(도구 목록)`: `AIMessage` 의 `tool_calls` 를 읽어 해당 도구를 실행하고, 결과를 담은 `ToolMessage` 를 반환합니다.
    * 도구는 실행 결과 자체를, `ToolNode` 는 결과를 포함한 `ToolMessage` 를 돌려준다는 점이 다릅니다.

In [ ]:
from langgraph.prebuilt import ToolNode

tool_list = [add, multiply]
llm_with_tools = llm.bind_tools(tool_list)  # 도구 설명이 포함된 LLM

tool_node = ToolNode(tool_list)  # 도구를 실행하는 노드

print(type(llm_with_tools))
print(type(tool_node))

In [ ]:
# LLM 에게 계산을 물으면 답 대신 "어떤 도구를 어떤 인자로 호출하라"는 tool_calls 를 만듦 (LLM 호출)
ai_message = llm_with_tools.invoke("3 더하기 5는 얼마인가요?")
ai_message.pretty_print()

In [ ]:
# tool_calls: [{'name': 도구 이름, 'args': 인자, 'id': 호출 ID, 'type': 'tool_call'}]
ai_message.tool_calls

* `ToolNode` 는 `tool_calls` 를 포함한 `AIMessage` 를 `{"messages": [...]}` 형태로 받습니다.
* 아래는 LLM 호출 없이 `tool_calls` 를 **직접 만들어** 전달하는 예입니다. LLM 응답과 무관하게 항상 같은 결과가 나오므로 `ToolNode` 의 동작만 확인할 수 있습니다.
* 주의: 현재 LangGraph 에서 `ToolNode` 는 그래프 실행 환경(runtime)이 필요해 `tool_node.invoke(...)` 를 **단독 호출하면 오류**(`Missing required config key`)가 납니다. `ToolNode` 하나만 넣은 최소 그래프로 감싸서 실행합니다.

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.graph import MessagesState, StateGraph, START, END

# LLM 이 만들었을 법한 tool_calls 를 직접 구성 (id 는 ToolMessage 와 짝을 맞추는 값)
manual_ai_message = AIMessage(
    content="",
    tool_calls=[{"name": "add", "args": {"a": 3, "b": 5}, "id": "call_demo_1", "type": "tool_call"}],
)

# ToolNode 하나만 가진 최소 그래프: START → tools → END
tool_only_graph = (
    StateGraph(MessagesState)
    .add_node("tools", tool_node)
    .add_edge(START, "tools")
    .add_edge("tools", END)
    .compile()
)

tool_msg = tool_only_graph.invoke({"messages": [manual_ai_message]})

# 결과 messages: [입력한 AIMessage, ToolMessage(content='8', name='add', tool_call_id='call_demo_1')]
tool_msg["messages"][-1].pretty_print()

`(4) StateGraph 로 에이전트 직접 구성`
* `MessagesState` 를 상태로 사용하고, `agent` 노드와 `tools` 노드를 **조건부 엣지**로 연결합니다.
* 이렇게 만든 구조를 `create_agent` 가 내부에서 대신 만들어 줍니다. 원리를 이해하기 위한 단계입니다.

In [ ]:
from langgraph.graph import MessagesState, StateGraph

graph_builder = StateGraph(MessagesState)

In [ ]:
def agent(state: MessagesState) -> MessagesState:
    """지금까지의 메시지를 도구가 연결된 LLM 에 전달해 응답(답변 또는 tool_calls)을 상태에 추가합니다."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

In [ ]:
from typing import Literal
from langgraph.graph import END


def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """마지막 AI 메시지에 도구 호출이 있으면 'tools', 없으면 END 로 가도록 경로를 결정합니다."""
    last_ai_message = state["messages"][-1]

    if last_ai_message.tool_calls:
        return "tools"  # 도구 실행이 필요함
    return END  # 최종 답변 완료

* `node` 를 추가하고 `edge` 로 연결합니다.

In [ ]:
graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", tool_node)  # 앞에서 만든 ToolNode 재사용

list(graph_builder.nodes)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_edge(START, "agent")
# 조건부 엣지: should_continue 의 반환값(키) → 이동할 노드(값). 키와 값이 같으면 그대로 이동
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)
graph_builder.add_edge("tools", "agent")  # 도구 실행 후 다시 agent 로 (루프)

# edges 에는 일반 엣지만, 조건부 엣지는 branches 에 보관됨
print("edges   :", graph_builder.edges)
print("branches:", dict(graph_builder.branches))

In [ ]:
graph = graph_builder.compile()
print(type(graph))
graph

In [ ]:
# 그래프 구조 확인: mermaid 코드 출력 (https://mermaid.live/ 에 붙여 넣으면 그림으로 보임)
mermaid_code = graph.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

In [ ]:
# 이미지로 직접 보려면: 
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


* https://mermaid.live/ 에서 `mermaid_code` 로 직접 확인할 수 있습니다.
* **점선(`-.->`)** 이 조건부 엣지입니다. `agent` 에서 `tools` 또는 `__end__` 로 갈 수 있고, `tools` → `agent` 실선이 루프를 만듭니다.

* `graph.stream()` 을 쓰면 에이전트가 답변을 만드는 과정(도구 호출 → 결과 → 최종 답)을 단계별로 볼 수 있습니다.

In [ ]:
from langchain_core.messages import HumanMessage

query = "3에다 5를 더한 결과에 다시 8을 곱하면 얼마인가요?"

# stream_mode="values": 단계마다 전체 상태를 내보냄 → 마지막 메시지만 출력
for chunk in graph.stream({"messages": [HumanMessage(query)]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

`(5) create_agent 로 한 줄 생성`
* `langchain.agents.create_agent` 는 위 (4)의 `agent` ⇄ `tools` 루프를 내부에서 `StateGraph` 로 자동 구성합니다.
* 예전의 `langgraph.prebuilt.create_react_agent` 는 **deprecated** 이므로 `create_agent` 를 사용합니다.
* `system_prompt` 는 에이전트의 역할 지시입니다. (선택 사항이지만 도구 사용 방침을 적어 두면 안정적입니다)

In [ ]:
from langchain.agents import create_agent

# 도구 목록과 시스템 프롬프트만 전달하면 에이전트 완성 (내부에서 StateGraph 자동 구성)
simple_agent = create_agent(
    model=llm,
    tools=tool_list,
    system_prompt="You are a helpful assistant with access to tools.",
)
print(type(simple_agent))

result = simple_agent.invoke({"messages": [HumanMessage(content=query)]})

# 전체 메시지 흐름 출력: Human → AI(tool_calls) → Tool → AI(최종 답)
for m in result["messages"]:
    m.pretty_print()

`(6) 복합 도구 실습`
* 성격이 다른 세 가지 도구(복리, 세금, 요일)를 주고 질문에 맞는 도구를 **스스로 선택**하는지 확인합니다.
* 한 질문에 도구가 여러 개 필요하면(마지막 질문) 에이전트가 도구를 **연달아** 호출합니다.

In [ ]:
from datetime import datetime


@tool
def calculate_compound_interest(principal: float, rate: float, years: int) -> str:
    """원금, 이자율(%), 기간(년)을 받아 복리 이자를 계산합니다."""
    amount = principal * (1 + rate / 100) ** years
    interest = amount - principal
    return f"{years}년 후 원리금: {amount:,.0f}원, 이자: {interest:,.0f}원"


@tool
def calculate_tax(income: float) -> str:
    """연소득을 받아 간이 세액을 계산합니다. (누진세율 구간별 계산)"""
    if income <= 12_000_000:
        tax = income * 0.06
    elif income <= 46_000_000:
        tax = 720_000 + (income - 12_000_000) * 0.15
    elif income <= 88_000_000:
        tax = 5_820_000 + (income - 46_000_000) * 0.24
    else:
        tax = 15_900_000 + (income - 88_000_000) * 0.35

    return f"예상 세액: {tax:,.0f}원, 세후 소득: {income - tax:,.0f}원"


@tool
def get_weekday(date_str: str) -> str:
    """YYYY-MM-DD 형식의 날짜를 받아 요일을 반환합니다."""
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    weekdays = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    return f"{date_str}은 {weekdays[date_obj.weekday()]}입니다."


finance_tools = [calculate_compound_interest, calculate_tax, get_weekday]

finance_agent = create_agent(
    model=llm,
    tools=finance_tools,
    system_prompt="당신은 도움이 되는 어시스턴트입니다. 사용자의 질문을 이해하고 적절한 도구를 사용하여 정확한 답변을 제공하세요.",
)
print(type(finance_agent))

In [ ]:
complex_questions = [
    "1억원을 연 5% 복리로 10년 동안 예금하면 얼마가 될까?",
    "연소득 5천만원일 때 내야 할 세금은 얼마야?",
    "2026년 1월 22일은 무슨 요일이야?",
    "연소득 8천만원일 때 세금을 계산하고, 남은 돈을 연 4% 복리로 5년간 예금하면 얼마가 돼?",
]

# 질문마다 최종 답변만 출력
for question in complex_questions:
    print(f"\n질문: {question}")
    print("-" * 50)

    result = finance_agent.invoke({"messages": [HumanMessage(content=question)]})
    print(f"답변: {result['messages'][-1].content}")
    print("=" * 80)

* 마지막 질문(도구 2개가 순서대로 필요)의 **중간 과정**을 `stream` 으로 확인합니다.

In [ ]:
# 도구 호출 순서(세금 → 복리)와 각 도구의 결과를 단계별로 확인
for chunk in finance_agent.stream({"messages": [HumanMessage(content=complex_questions[-1])]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()